# 🤖 Model Experiments — ChurnGuard AI

Compare **Logistic Regression**, **Random Forest** and **XGBoost** on the engineered BigML Telecom Churn data. The data is pre-split 80/20, so we train on the 80% file and evaluate on the held-out 20% file.

The production training routine lives in `src/train_model.py`; here we explore the comparison interactively (metrics, confusion matrices, ROC curves, feature importance, threshold tuning).

**Why not accuracy alone?** The data is imbalanced (~14% churn), so we focus on **precision, recall, F1 and ROC-AUC**.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.data_preprocessing import load_clean_split
from src.feature_engineering import add_features
from src.train_model import build_preprocessor, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from src.evaluate import evaluate_model, format_metrics

plt.rcParams['figure.figsize'] = (8, 5)

## 1. Prepare the data

Load + clean both splits, engineer features, then fit the preprocessor on **train only** to avoid leakage.

In [ ]:
train, test = load_clean_split()
train, test = add_features(train), add_features(test)

y_train = train[config.TARGET_COLUMN]
X_train = train.drop(columns=[config.TARGET_COLUMN])
y_test = test[config.TARGET_COLUMN]
X_test = test.drop(columns=[config.TARGET_COLUMN])

pre = build_preprocessor()
X_train_t = pre.fit_transform(X_train)
X_test_t = pre.transform(X_test)
feature_names = pre.get_feature_names_out().tolist()

neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos
print(f'Train: {X_train_t.shape} | Test: {X_test_t.shape} | features: {len(feature_names)}')
print(f'Class balance -> neg={neg}, pos={pos}, scale_pos_weight={scale_pos_weight:.2f}')

## 2. Define the candidate models

All three are configured to handle class imbalance (class weights / `scale_pos_weight`).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_depth=12, class_weight='balanced',
        random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.9,
        colsample_bytree=0.9, scale_pos_weight=scale_pos_weight,
        eval_metric='logloss', random_state=42, n_jobs=-1),
}

## 3. Train & compare

In [ ]:
results, fitted, probas = {}, {}, {}
for name, model in models.items():
    model.fit(X_train_t, y_train)
    proba = model.predict_proba(X_test_t)[:, 1]
    pred = model.predict(X_test_t)
    results[name] = evaluate_model(y_test.to_numpy(), pred, proba)
    fitted[name], probas[name] = model, proba
    print(format_metrics(name, results[name]))

In [ ]:
# Side-by-side comparison table
comparison = pd.DataFrame(results).T[
    ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']]
comparison.style.highlight_max(axis=0, color='#bbf7d0') if hasattr(
    comparison, 'style') else comparison

In [ ]:
ax = comparison[['precision', 'recall', 'f1', 'roc_auc']].plot(
    kind='bar', figsize=(10, 5))
ax.set_title('Model comparison (test set)'); ax.set_ylabel('Score')
ax.set_ylim(0, 1); ax.legend(loc='lower right')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 4. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, results):
    cm = np.array(results[name]['confusion_matrix'])
    im = ax.imshow(cm, cmap='Blues')
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha='center', va='center',
                color='white' if v > cm.max() / 2 else 'black', fontsize=13)
    ax.set_title(name); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Retained', 'Churn'])
    ax.set_yticks([0, 1]); ax.set_yticklabels(['Retained', 'Churn'])
plt.tight_layout(); plt.show()

## 5. ROC curves

In [ ]:
from sklearn.metrics import roc_curve, auc

for name, proba in probas.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc(fpr, tpr):.3f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.4)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC curves'); plt.legend(loc='lower right')
plt.tight_layout(); plt.show()

## 6. XGBoost feature importance (top 15)

In [ ]:
xgb = fitted['XGBoost']
imp = pd.Series(xgb.feature_importances_, index=feature_names)
imp.sort_values().tail(15).plot(kind='barh', color='#2563eb',
                                 title='XGBoost feature importance (top 15)')
plt.tight_layout(); plt.show()

## 7. Threshold tuning (business trade-off)

The default 0.5 cut-off is rarely optimal for an imbalanced retention problem. Catching more churners (higher recall) usually costs precision. Below we inspect the trade-off for XGBoost.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

proba = probas['XGBoost']
rows = []
for t in np.arange(0.1, 0.91, 0.1):
    pred = (proba >= t).astype(int)
    rows.append({'threshold': round(t, 2),
                 'precision': precision_score(y_test, pred, zero_division=0),
                 'recall': recall_score(y_test, pred, zero_division=0),
                 'f1': f1_score(y_test, pred, zero_division=0)})
thr = pd.DataFrame(rows).set_index('threshold')
display(thr.round(3))
thr.plot(marker='o', title='Precision / Recall / F1 vs decision threshold')
plt.xlabel('Threshold'); plt.tight_layout(); plt.show()

## 8. (Reference) SMOTE vs class_weight

We also tried **SMOTE** (synthetic oversampling of the minority class), applied to the **training matrix only** — never the test set. Requires `pip install imbalanced-learn`.

> **Decision:** production keeps `class_weight` / `scale_pos_weight` — SMOTE's gain was marginal and avoids an extra dependency. Kept here for reference.
> Note: SMOTE here runs on one-hot-encoded data; `SMOTENC` would be more correct for categoricals.

In [ ]:
from imblearn.over_sampling import SMOTE

X_res, y_res = SMOTE(random_state=42).fit_resample(X_train_t, y_train)
print('Before SMOTE:', np.bincount(y_train), '-> after:', np.bincount(y_res))

smote_models = {
    'LogReg + SMOTE': LogisticRegression(max_iter=1000, random_state=42),
    'RF + SMOTE': RandomForestClassifier(n_estimators=300, max_depth=12,
                                         random_state=42, n_jobs=-1),
    'XGB + SMOTE': XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05,
                                 subsample=0.9, colsample_bytree=0.9,
                                 eval_metric='logloss', random_state=42, n_jobs=-1),
}
smote_results = {}
for name, model in smote_models.items():
    model.fit(X_res, y_res)
    proba = model.predict_proba(X_test_t)[:, 1]
    pred = model.predict(X_test_t)
    smote_results[name] = evaluate_model(y_test.to_numpy(), pred, proba)
    print(format_metrics(name, smote_results[name]))

In [ ]:
# class_weight vs SMOTE side by side
cw = comparison[['precision', 'recall', 'f1', 'roc_auc']].copy()
cw.index = [f'{i} (cw)' for i in cw.index]
sm = pd.DataFrame(smote_results).T[['precision', 'recall', 'f1', 'roc_auc']]
pd.concat([cw, sm]).round(3)

## 9. Conclusion

- **XGBoost** gives the best balance of precision, recall and F1 and is selected as the **production model** in `src/train_model.py`.
- Random Forest is competitive; Logistic Regression trades precision for recall.
- **SMOTE** (section 8) was tested but gave only a marginal gain over `class_weight`, so production keeps the simpler class-weighting approach.
- The risk thresholds in `config.RISK_THRESHOLDS` (Low/Medium/High) map the predicted probability to actionable segments — see the threshold-tuning chart for the business trade-off.

Re-run the full, reproducible training (which also saves the pipeline and SHAP explainer) with:

```python
from src.train_model import train
train()
```